# 12.3 Coding Practice: Lost in Translation? Entities Across Languages (Solution)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module12/week12_multilingual_entities_practice_solution.ipynb)

**Ungraded · about 35–45 minutes · CPU only**

**Solution.** Completed TODOs, suggested answers, and notes on the outputs we observed when this notebook was tested. Try the practice notebook yourself before reading this one.

This activity follows directly from Reading 12.1 (*Multilingual Named Entity Recognition*) and Reading 12.2 (*Machine Translation*). Both readings build toward one example: the word **Amazon**, which is an easy organization (ORG) tag in English but a genuine problem the moment translation or a different language enters the picture. Here you will use real pretrained models to see where that problem shows up and where it does not.


## Learning goals

- Run a pretrained multilingual NER model and read its entity-type and confidence output.
- Find a sentence where the NER model's prediction is ambiguous or outright wrong, and explain why.
- Run a pretrained translation model and compare its output against the original for adequacy and fluency.
- Translate a sentence containing a named entity, then re-tag the translation, to see whether translation changed the entity's type or boundaries.
- Connect each observation back to a specific idea from 12.1 or 12.2.


## Setup

Choose a standard **CPU** runtime in Colab (Runtime > Change runtime type > CPU). Run the cells from top to bottom.

This notebook uses two public models from the Hugging Face Hub:

- `Davlan/xlm-roberta-base-ner-hrl` for named entity recognition (about 1.1 GB, downloaded once and cached).
- `Helsinki-NLP/opus-mt-en-de` for English-to-German translation (about 300 MB, downloaded once and cached).

No Hugging Face account, token, or API key is needed. The first run takes a minute or two to download both models; later runs in the same session reuse the cache.

Both models are **permissively licensed** for this kind of use: the NER model is AFL-3.0, and the translation model is CC-BY-4.0.


In [ ]:
%pip -q install "transformers==5.17.0" "sentencepiece==0.2.1" "protobuf==6.33.1" "sacremoses==0.1.1"

Now import the libraries and load both models. We load each model **once** and reuse it for every example below, the same pattern as Weeks 9 and 10.

A note on the translation model: this version of the `transformers` library does not include a ready-made `pipeline("translation", ...)` shortcut the way it does for NER or sentiment. We instead load the tokenizer and the model directly and call `.generate()` ourselves. This is normal; a pipeline is a convenience wrapper over exactly these two steps, and not every task has one in every library version.


In [2]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

transformers.logging.set_verbosity_error()  # hide long loading messages
torch.set_num_threads(2)
print("Transformers:", transformers.__version__, "| PyTorch:", torch.__version__)

NER_MODEL = "Davlan/xlm-roberta-base-ner-hrl"
MT_MODEL = "Helsinki-NLP/opus-mt-en-de"

ner = pipeline(
    "ner",
    model=NER_MODEL,
    tokenizer=NER_MODEL,
    aggregation_strategy="simple",  # merge sub-word pieces into one entity per span
    device=-1,  # CPU
    model_kwargs={"disable_mmap": True},  # load weights into memory for CPU compatibility
)

mt_tokenizer = AutoTokenizer.from_pretrained(MT_MODEL)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL, disable_mmap=True)
mt_model.eval()


def translate(text, max_new_tokens=60):
    """Translate one English sentence to German."""
    batch = mt_tokenizer([text], return_tensors="pt", padding=True)
    with torch.inference_mode():
        generated = mt_model.generate(**batch, max_new_tokens=max_new_tokens)
    return mt_tokenizer.batch_decode(generated, skip_special_tokens=True)[0]


def show_entities(text):
    """Print each entity the NER model finds, with its type and confidence."""
    print(text)
    entities = ner(text)
    if not entities:
        print("  (no entities found)")
    for entity in entities:
        print(f"  {entity['entity_group']:4s} '{entity['word']}'  score={entity['score']:.3f}")
    return entities


Transformers: 5.17.0 | PyTorch: 2.14.0


## Part 1: Multilingual Named Entity Recognition

`Davlan/xlm-roberta-base-ner-hrl` is a version of **XLM-RoBERTa**, the exact multilingual transformer Reading 12.1 Section 4 names as the standard way to handle many languages with one model, already fine-tuned for NER in 10 languages. Instead of fine-tuning it ourselves (what the legacy course notebook for this week does, with a GPU and several hours), we use someone else's completed fine-tuning directly, the "ready-made model" option from Week 10's three ways to use a pretrained model.

One difference from Reading 12.1's four-category table: this particular model only uses three tags, **PER**, **ORG**, and **LOC** (no **GPE**). Real pretrained models do not always match a textbook's exact taxonomy, which is itself worth noticing.


### Guided Example 1

Run the cell below. It tags a sentence that contains all three entity types this model knows about.

In [3]:
_ = show_entities("Jeff Bezos founded Amazon in Seattle in 1994.")

Jeff Bezos founded Amazon in Seattle in 1994.


  PER  'Jeff Bezos'  score=1.000
  ORG  'Amazon'  score=1.000
  LOC  'Seattle'  score=0.999


Three entities, three types: a person (`PER`), an organization (`ORG`), and a location (`LOC`). The `score` is the model's confidence, as a probability, in its own prediction. Now compare that with a sentence about the **other** Amazon:

In [4]:
_ = show_entities("Scientists are studying biodiversity loss in the Amazon rainforest.")

Scientists are studying biodiversity loss in the Amazon rainforest.
  LOC  'Amazon'  score=1.000


Same word, different type: the model correctly reads *Amazon* as `LOC` here, because "rainforest" gives it enough context, exactly the kind of disambiguation Reading 12.1 Section 2 describes. The model is not matching the string "Amazon" to a fixed label; it is using the surrounding words.


### Checkpoint 1

The model does not always get it right. Run it on the sentence below.

**TODO 1:** Before running the next cell, predict: will the model tag *Amazon* correctly here, incorrectly, or not at all?


In [5]:
# TODO 1: run this cell, then compare the result with your prediction above.
_ = show_entities("I saw an Amazon parrot at the zoo.")

I saw an Amazon parrot at the zoo.
  LOC  'Amazon'  score=0.957


**Question 1:** What type did the model assign to *Amazon* here, and why is that a genuine mistake rather than a defensible judgment call? (Hint: is "Amazon parrot" naming a specific place or company, or is it an ordinary noun phrase for a kind of bird?) Then try your own sentence below, using a word or phrase you expect the model to find hard, and explain what you found.

**Suggested answer:** In our test, the model tagged *Amazon* as `LOC` with high confidence (about 0.96). That is a genuine mistake, not a defensible judgment call: "Amazon parrot" is an ordinary noun phrase naming a kind of bird (the genus *Amazona*), not a reference to the company or the river at all. The correct answer here is **no entity**. The model likely latched onto "Amazon" plus a plausible-sounding context word and defaulted to its most common geographic association, the same kind of shortcut behind the Checkpoint 1 and Part 3 mistakes compared in the reflection questions below.


In [6]:
# Try your own sentence here.
my_sentence = "Amazon is huge."
_ = show_entities(my_sentence)

Amazon is huge.


  LOC  'Amazon'  score=0.967


**Suggested answer (our own sentence):** With almost no context, "Amazon is huge." gets tagged `LOC` (about 0.97 confidence) rather than `ORG`. Out of context, *huge* fits a river or rainforest about as well as a company, so the model's choice here reflects a real ambiguity, not a clear error. This is a useful contrast with the parrot sentence: there, the sentence had enough context to settle the question (it just settled it wrong); here, the sentence itself does not give the model, or a human reader, enough to go on.

## Part 2: Machine Translation

`Helsinki-NLP/opus-mt-en-de` is a small encoder-decoder transformer, the same architecture family Reading 12.2 Section 3 describes, trained specifically for English-to-German translation. It is much smaller than a production translation system, but it is built the same way.


### Guided Example 2

Reading 12.2 Section 1 pointed out that translating *I saw the movie and it is good* into a language with grammatical gender forces a choice English does not. Run the cell below and see what the model chose for *it*.

In [7]:
english = "I saw the movie and it is good."
german = translate(english)
print("EN:", english)
print("DE:", german)

EN: I saw the movie and it is good.
DE: Ich habe den Film gesehen und er ist gut.


The model had to pick a German pronoun for *it*, referring back to *der Film* (masculine). Nothing in the English sentence told it to do that; it is a requirement of German grammar, exactly as the reading describes. Now look at an idiom:


In [8]:
english = "Don't spill the beans about the surprise party."
german = translate(english)
print("EN:", english)
print("DE:", german)

EN: Don't spill the beans about the surprise party.
DE: Verschütten Sie die Bohnen nicht wegen der Überraschungsparty.


This comes out **literal**, not idiomatic: the German sentence talks about actually spilling beans, which is exactly the "word for word is nonsense" problem Reading 12.2 Section 1 describes for idioms. A fluent human translator would reach for a German idiom with the same meaning instead.


### Checkpoint 2

**TODO 2:** Change the sentence below to one of your own (a different idiom, a sentence with grammatical gender, or anything else from Reading 12.2 Section 1), and compare the translation to the original.


In [9]:
# TODO 2: replace the sentence below with your own, then run this cell.
my_sentence = "He kicked the bucket last winter."
print("EN:", my_sentence)
print("DE:", translate(my_sentence))

EN: He kicked the bucket last winter.


DE: Er hat letzten Winter in den Eimer getreten.


**Question 2:** Rate your translation on the two dimensions from Reading 12.2 Section 4: **adequacy** (how much of the original meaning survives) and **fluency** (how natural the German reads). Did you find a case where the two disagree, a fluent sentence that changes the meaning, or an awkward sentence that keeps it?

**Suggested answer:** In our test, "He kicked the bucket last winter." came back as a literal translation about an actual bucket ("Er hat letzten Winter in den Eimer getreten."), not the German idiom for dying (*den Löffel abgeben*, literally "to hand over the spoon"). **Fluency:** the German sentence is perfectly grammatical and reads smoothly. **Adequacy:** it is wrong; the original meaning (someone died) is completely lost. This is exactly the disagreement the question asks about: a fluent sentence that is not adequate. A human rater checking only "does this sound natural in German" could easily miss that the meaning changed.


## Part 3: Connecting NER and Translation

Reading 12.2 Section 5 makes a specific claim: a translation error can quietly become an information-extraction error, because a mangled name is no longer the name an NER tagger expects to see. We can test this directly, with the same two models, by tagging a sentence, translating it, and tagging the translation.


### Guided Example 3

The helper below runs all three steps and prints the entities found before and after translation.

In [10]:
def compare_before_after(sentence):
    print("Original:", sentence)
    before = show_entities(sentence)
    german = translate(sentence)
    print()
    print("Translated:", german)
    after = show_entities(german)
    return before, after


_ = compare_before_after("Amazon reported strong quarterly earnings this year.")

Original: Amazon reported strong quarterly earnings this year.
Amazon reported strong quarterly earnings this year.


  ORG  'Amazon'  score=1.000


Translated: Amazon meldete in diesem Jahr ein starkes Quartalsergebnis.
Amazon meldete in diesem Jahr ein starkes Quartalsergebnis.
  ORG  'Amazon'  score=1.000


Here the entity survives the trip: *Amazon* stays `ORG` on both sides, and the translation keeps the brand name unchanged, exactly as Reading 12.2 Section 5 says a brand name should be handled. Now try a sentence that is almost the same, but about a different kind of Amazon fact:


In [11]:
_ = compare_before_after("Workers at the Amazon warehouse in Ohio went on strike.")

Original: Workers at the Amazon warehouse in Ohio went on strike.
Workers at the Amazon warehouse in Ohio went on strike.
  ORG  'Amazon'  score=0.999
  LOC  'Ohio'  score=1.000


Translated: Arbeiter im Amazona-Lagerhaus in Ohio streikten.
Arbeiter im Amazona-Lagerhaus in Ohio streikten.
  LOC  'Amazona-Lagerhaus'  score=0.964
  LOC  'Ohio'  score=1.000


This time something breaks. Look closely at the German translation: the model wrote **"Amazona-Lagerhaus"**, not "Amazon-Lagerhaus". The translation model itself introduced an error, turning the company name into something that looks more like the river's Spanish or Portuguese form (*Amazonas*). That mangled name then confuses the NER tagger: it no longer recognizes an organization, and tags the whole compound as a **location** instead. This is exactly the chain Reading 12.2 Section 5 describes: a translation error becoming an information-extraction error, and it happened with real pretrained models, not a hypothetical.

Compare this with the rainforest sentence, where nothing goes wrong on either side:


In [12]:
_ = compare_before_after("Scientists are studying biodiversity loss in the Amazon rainforest.")

Original: Scientists are studying biodiversity loss in the Amazon rainforest.
Scientists are studying biodiversity loss in the Amazon rainforest.
  LOC  'Amazon'  score=1.000


Translated: Wissenschaftler untersuchen den Biodiversitätsverlust im Amazonas-Regenwald.
Wissenschaftler untersuchen den Biodiversitätsverlust im Amazonas-Regenwald.
  LOC  'Amazonas-Regenwald'  score=0.983


Here *Amazon* becomes *Amazonas* too, but correctly: "Amazonas-Regenwald" is the standard German term for the Amazon rainforest, so the entity type stays `LOC` on both sides. The same kind of change (adding "Amazonas") was wrong in the warehouse sentence and right here, because the underlying entity really is different: one is a brand that should never change, the other is a place with its own name in German.


### Checkpoint 3

**TODO 3:** Pick one sentence, either one of the three above or one of your own containing a named entity, and run it through `compare_before_after`.


In [13]:
# TODO 3: pick a sentence (reuse one above or write your own) and run it through the pipeline.
my_sentence = "The Amazon stretches across nine countries in South America."
_ = compare_before_after(my_sentence)

Original: The Amazon stretches across nine countries in South America.
The Amazon stretches across nine countries in South America.
  LOC  'Amazon'  score=0.999
  LOC  'South America'  score=1.000


Translated: Der Amazonas erstreckt sich über neun Länder in Südamerika.
Der Amazonas erstreckt sich über neun Länder in Südamerika.
  LOC  'Amazonas'  score=0.693
  LOC  'Südamerika'  score=0.989


**Question 3:** Did the entity's type or boundaries change between the English and German versions? If something changed, was the translation itself wrong (like the warehouse example), or did the entity correctly take a different conventional form (like the rainforest example)? Explain which one happened and how you can tell the difference.

**Suggested answer:** In our test, "The Amazon stretches across nine countries in South America." tagged *Amazon* as `LOC` in English, and the German translation ("Der Amazonas erstreckt sich über neun Länder in Südamerika.") tagged *Amazonas* as `LOC` too, though with a noticeably lower confidence (about 0.69 versus 0.999 in English). Nothing broke: the entity's type stayed the same on both sides, even if the model hedged more about it in German. This is the "correctly took a different conventional form" case, the same pattern as the rainforest example, not the warehouse example's translation error. The way to tell the difference is to check whether the *change itself* makes sense for what the entity actually is: a river correctly gaining a German place-name form is not an error, but a company brand name gaining the same form would be.


## Reflection

1. Reading 12.1 Section 4 describes **zero-shot cross-lingual transfer**: fine-tune on one language, apply to others never seen in training. The NER model you used was explicitly fine-tuned on all 10 of its languages, not zero-shot. What would you need to check, that you could not check just by trying examples in this notebook, to know whether a model is really transferring zero-shot rather than simply having been trained on everything you are testing it on?
2. Reading 12.2 Section 4 describes BLEU as a word-overlap metric, not a meaning metric. Think about the warehouse sentence's translation ("Amazona-Lagerhaus" instead of "Amazon-Lagerhaus"), a one-word difference. Would you expect a word-overlap metric like BLEU to catch this error clearly, or could a one-word change hide inside an otherwise high overlap score? What did catching the error in this notebook actually depend on?
3. Look back at Checkpoint 1's "Amazon parrot" mistake and Part 3's "Amazona-Lagerhaus" mistake. Both are the same underlying model family making an error on the same word, *Amazon*, in two different tasks. What do the two mistakes have in common, and what is different about how each one happened?

```{dropdown} Suggested answers

1. You would need to check that the language you are testing was genuinely absent from the model's NER fine-tuning data, not just that you personally did not fine-tune it yourself. Since this model's card states it was fine-tuned on all 10 of its languages, nothing you observed in this notebook is zero-shot transfer; it is in-domain performance. A real zero-shot test would need a labeled test set in an eleventh language the model's training data never covered, and comparing its F1 there against a language it was trained on (the same comparison 12.1's discussion question 4 asks about Swahili).
2. A one-word change like "Amazon" to "Amazona" would likely leave most of the sentence's other words and word-sequences intact, so BLEU's overlap-counting could still score the sentence fairly high overall; BLEU does not single out named entities for extra scrutiny. Catching the error in this notebook depended on actually reading the German output closely and comparing it against the specific expectation that a brand name should not change, not on any automatic score.
3. Both mistakes involve the model treating "Amazon" according to its most statistically common association rather than the specific sense required by context: a geographic reading in the parrot sentence (where "Amazon" is part of a common noun phrase, not a proper name at all), and a geographic-sounding spelling change in the warehouse sentence (where translation defaulted toward the river/region's conventional form). The difference is which model made the error and at which step: Checkpoint 1's error is an NER model misreading English directly; Part 3's error is a translation model mangling the word first, which then causes a second, downstream NER error on the German text.
```



## Wrap-up

You used two independently pretrained models, one for named entity recognition and one for translation, and found that the places where they struggle are not random: they cluster around exactly the kind of ambiguity Readings 12.1 and 12.2 describe, a word that is sometimes a name and sometimes not, and a name that sometimes should change form across languages and sometimes should not. Neither model was trained for this course or tuned to produce these specific examples; the errors you saw are the models' own.


## Validation checks

These checks confirm that the notebook ran from top to bottom. If one fails, rerun the notebook from the start (Runtime > Restart and run all in Colab).


In [14]:
amazon_org = show_entities("Amazon reported strong quarterly earnings this year.")
assert any(e["entity_group"] == "ORG" and e["word"] == "Amazon" for e in amazon_org)

amazon_loc = show_entities("Scientists are studying biodiversity loss in the Amazon rainforest.")
assert any(e["entity_group"] == "LOC" for e in amazon_loc)

de_test = translate("Munich is a beautiful city in Germany.")
assert "München" in de_test or "Munchen" in de_test or "Munich" in de_test
print("All checks passed.")

Amazon reported strong quarterly earnings this year.
  ORG  'Amazon'  score=1.000
Scientists are studying biodiversity loss in the Amazon rainforest.
  LOC  'Amazon'  score=1.000


All checks passed.
